In [98]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col
import numpy as np
from statsmodels.stats.diagnostic import het_breuschpagan

url = 'https://raw.githubusercontent.com/git-iush/large_family/refs/heads/main/data2.parquet'

In [99]:
df = pd.read_parquet(url)
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [100]:
df.columns

Index(['SERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'PERNUM', 'PERWT', 'NCHILD',
       'SEX', 'AGE', 'MARST', 'HCOVANY', 'HINSCAID', 'EDUCD', 'INCTOT',
       'FTOTINC', 'INCWELFR', 'CONNECTICUT', 'MAINE', 'MASSACHUSETTS',
       'NEWHAMPSHIRE', 'RHODEISLAND', 'VERMONT', 'DELAWARE', 'NEWJERSEY',
       'NEWYORK', 'PENNSYLVANIA', 'ILLINOIS', 'INDIANA', 'MICHIGAN', 'OHIO',
       'WISCONSIN', 'IOWA', 'KANSAS', 'MINNESOTA', 'MISSOURI', 'NEBRASKA',
       'NORTHDAKOTA', 'SOUTHDAKOTA', 'VIRGINIA', 'ALABAMA', 'ARKANSAS',
       'GEORGIA', 'LOUISIANA', 'MISSISSIPPI', 'NORTHCAROLINA', 'SOUTHCAROLINA',
       'TEXAS', 'KENTUCKY', 'MARYLAND', 'OKLAHOMA', 'TENNESSEE',
       'WESTVIRGINIA', 'ARIZONA', 'COLORADO', 'IDAHO', 'MONTANA', 'NEVADA',
       'NEWMEXICO', 'UTAH', 'WYOMING', 'CALIFORNIA', 'OREGON', 'WASHINGTON',
       'ALASKA', 'HAWAII', 'DC', 'STATENAME', 'BLACK', 'AIAN', 'API', 'OTHER',
       'MULTR', 'HISP', 'EXPANSION', 'WELFARE', 'PRSNINC', 'YRSEDUC'],
      dtype='str')

In [101]:
medicaid_mapping = {1:0, 2:1}
df['MEDICAID'] = df['HINSCAID'].map(medicaid_mapping)
df['MEDFARE'] = df['MEDICAID'] * df['WELFARE']

df['FAMINC'] = df['FTOTINC'] - df['INCTOT']
df['PRSNINC'] = df['PRSNINC'] / 10000
df['FAMINC'] = df['FAMINC'] / 10000
df['PRSNINC2'] = df['PRSNINC'] ** 2
df['FAMINC2'] = df['FAMINC'] ** 2

df['AGE2'] = df['AGE'] ** 2

df['HIGHSCHOOL'] = [1 if i == 63 or i==64 else 0 for i in df['EDUCD']]
df['BACHELORS'] = [1 if i == 101 else 0 for i in df['EDUCD']]
df['ADVANCED'] = [1 if i>=114 else 0 for i in df['EDUCD']]

df['MARD'] = [1 if i==1 or i==2 or i==3 else 0 for i in df['MARST']]

df['LARGEFAM'] = [1 if i >= 3 else 0 for i in df['NCHILD']]

In [102]:
ols_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + BLACK + AIAN + API + OTHER + MULTR + HISP + EXPANSION + HIGHSCHOOL + BACHELORS + ADVANCED'
ols_fe = model_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + BLACK + AIAN + API + OTHER + MULTR + HISP + C(STATENAME, Treatment("FLORIDA")) + HIGHSCHOOL + BACHELORS + ADVANCED'

In [103]:
reg_exp = smf.ols(ols_exp, data=df).fit()
reg_fe = smf.ols(ols_fe, data=df).fit()

In [104]:
print(reg_exp.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.317
Model:                            OLS   Adj. R-squared:                  0.317
Method:                 Least Squares   F-statistic:                 1.571e+04
Date:                Mon, 16 Nov 2026   Prob (F-statistic):               0.00
Time:                        18:36:51   Log-Likelihood:            -9.6631e+05
No. Observations:              678476   AIC:                         1.933e+06
Df Residuals:                  678455   BIC:                         1.933e+06
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7519      0.014   -189.889      0.0

In [105]:
print(reg_fe.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.320
Model:                            OLS   Adj. R-squared:                  0.320
Method:                 Least Squares   F-statistic:                     4636.
Date:                Mon, 16 Nov 2026   Prob (F-statistic):               0.00
Time:                        18:36:51   Log-Likelihood:            -9.6441e+05
No. Observations:              678476   AIC:                         1.929e+06
Df Residuals:                  678406   BIC:                         1.930e+06
Df Model:                          69                                         
Covariance Type:            nonrobust                                         
                                                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------

states = df['STATENAME'].unique()

expreg = {}

for s in states:
    ols_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARST + BLACK + AIAN + API + OTHER + MULTR + HISP + EXPANSION + HIGHSCHOOL + BACHELORS + ADVANCED'

    ols = smf.ols(ols_exp, data=df[df['STATENAME']==s]).fit()
    expreg[s] = ols

results = summary_col(
    results = [reg_exp] + [expreg[i] for i in states],
    model_names = ['All'] + [i.capitalize() for i in states],
    info_dict = {'No. obs': lambda x: f'{int(x.nobs)}'},
    stars=True,
    regressor_order = ['Intercept', 'MEDFARE'],)

results.tables[0].drop('R-squared Adj.', inplace=True)
results.tables[0] = results.tables[0].transpose()
print(results)

In [108]:
results = summary_col(
    [reg_exp, reg_fe],
    model_names=['Expansion', 'One-Way State Fixed Effects'],
    regressor_order=['MEDICAID', 'WELFARE', 'MEDFARE', 'EXPANSION', 'AGE', 'AGE2', 'PRSNINC', 'PRSNINC2', 'FAMINC', 'FAMINC2'],
    stars=True,
    float_format='%0.5f',
    info_dict={
        'Observations': lambda x: f'{int(x.nobs):,}'
    },
    drop_omitted=True
)

print(results)


                Expansion  One-Way State Fixed Effects
------------------------------------------------------
MEDICAID       0.31545***  0.32748***                 
               (0.00349)   (0.00351)                  
WELFARE        -0.32724*** -0.30899***                
               (0.02196)   (0.02190)                  
MEDFARE        0.61130***  0.59881***                 
               (0.02447)   (0.02440)                  
EXPANSION      -0.02219***                            
               (0.00280)                              
AGE            0.17777***  0.17901***                 
               (0.00093)   (0.00093)                  
AGE2           -0.00209*** -0.00211***                
               (0.00001)   (0.00001)                  
PRSNINC        -0.02794*** -0.02652***                
               (0.00041)   (0.00041)                  
PRSNINC2       0.00044***  0.00042***                 
               (0.00001)   (0.00001)                  
FAMINC   

print(df[['PRSNINC', 'FAMINC']].describe(
    percentiles=[.01, .25, .50, .75, .99]
))

print(df[['PRSNINC', 'FAMINC']].head(10))